# DSN Mart Sales Prediction 

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

RANDOM_SEEDS = (2026, 2027)
N_SPLITS = 5
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set this after attaching the private dataset with v22.csv, v36.csv, v37.csv, and v42.csv.
COMPONENT_DIR = Path('/kaggle/input/dsn-mart-v47-components')

# Kaggle path first; local fallback makes the notebook runnable outside Kaggle as well.
candidate_dirs = [
    Path('/kaggle/input/dsn-bootcamp-qualification-hackathon-2026-ml-track'),
    Path.home() / 'Downloads' / 'dsn-bootcamp-qualification-hackathon-2026-ml-track',
]
DATA_DIR = next((p for p in candidate_dirs if (p / 'train.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Set DATA_DIR to the folder containing train.csv and test.csv.')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
print(train.shape, test.shape)

(6818, 13) (1705, 12)


## Feature preparation

All engineered columns below use only predictors. Product-level means and counts are calculated from the combined train/test predictors, which is safe because no target values are involved.

In [2]:
def clean_text(series):
    return series.fillna('missing').astype(str).str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def build_features(train_df, test_df):
    tr = train_df.drop(columns=['total_sales']).copy()
    both = pd.concat([tr, test_df.copy()], ignore_index=True)
    both['fat_content_clean'] = clean_text(both['fat_content'])
    both['product_category_clean'] = clean_text(both['product_category'])
    both['store_size_clean'] = clean_text(both['store_size'])

    product_weight = both.groupby('product_code')['product_weight_kg'].transform('median')
    category_weight = both.groupby('product_category_clean')['product_weight_kg'].transform('median')
    both['product_weight_imputed'] = both['product_weight_kg'].fillna(product_weight).fillna(category_weight)
    both['missing_weight'] = both['product_weight_kg'].isna().astype('int8')
    both['zero_visibility'] = (both['shelf_visibility'] == 0).astype('int8')
    both['log_visibility'] = np.log1p(both['shelf_visibility'])

    product_vis = both.groupby('product_code')['shelf_visibility'].transform('mean')
    product_price = both.groupby('product_code')['product_price'].transform('mean')
    both['visibility_rel_product'] = both['shelf_visibility'] / (product_vis + 1e-6)
    both['price_rel_product'] = both['product_price'] / (product_price + 1e-6)
    both['product_obs_count'] = both.groupby('product_code')['id'].transform('size')
    both['store_obs_count'] = both.groupby('store_code')['id'].transform('size')

    both = both.drop(columns=['id', 'fat_content', 'product_category', 'store_size'])
    for column in both.select_dtypes(include=['object', 'string']).columns:
        both[column] = both[column].fillna('missing').astype(str)
    return both.iloc[:len(train_df)].reset_index(drop=True), both.iloc[len(train_df):].reset_index(drop=True)

X, X_test = build_features(train, test)
y = train['total_sales'].astype(float).to_numpy()
cat_features = [i for i, c in enumerate(X.columns) if X[c].dtype == object or str(X[c].dtype).startswith('string')]
print(X.shape, len(cat_features))

(6818, 19) 0


## Cross-validated backup models

Three complementary targets are used:

- **Direct:** raw total sales.
- **Demand:** total sales divided by product price, with price-squared training weights.
- **Store-normalized:** sales standardized within each outlet using training-fold statistics only.

Each prediction is generated from a model that did not train on that validation row.

In [3]:
def cat_model(seed):
    return CatBoostRegressor(
        loss_function='RMSE', eval_metric='RMSE', iterations=2600,
        learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
        random_strength=0.7, bagging_temperature=0.5,
        random_seed=seed, od_type='Iter', od_wait=150,
        verbose=False, allow_writing_files=False
    )

def train_family(kind):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    for seed in RANDOM_SEEDS:
        fold_oof = np.zeros(len(X))
        fold_test = np.zeros(len(X_test))
        splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for fold, (fit, valid) in enumerate(splitter.split(X), start=1):
            if kind == 'direct':
                target_fit, target_valid = y[fit], y[valid]
                fit_weight = None
                restore = lambda p, frame: p
            elif kind == 'demand':
                target_fit = y[fit] / X.iloc[fit]['product_price'].to_numpy()
                target_valid = y[valid] / X.iloc[valid]['product_price'].to_numpy()
                fit_weight = X.iloc[fit]['product_price'].to_numpy() ** 2
                restore = lambda p, frame: p * frame['product_price'].to_numpy()
            elif kind == 'store_norm':
                stats = train.iloc[fit].assign(target=y[fit]).groupby('store_code')['target'].agg(['mean', 'std'])
                fit_store = train.iloc[fit]['store_code'].map(stats['mean']).to_numpy()
                fit_std = train.iloc[fit]['store_code'].map(stats['std']).fillna(1).to_numpy()
                target_fit = (y[fit] - fit_store) / fit_std
                target_valid = (y[valid] - train.iloc[valid]['store_code'].map(stats['mean']).to_numpy()) / train.iloc[valid]['store_code'].map(stats['std']).fillna(1).to_numpy()
                fit_weight = None
                restore = lambda p, frame: p * frame['store_code'].map(stats['std']).fillna(1).to_numpy() + frame['store_code'].map(stats['mean']).fillna(y[fit].mean()).to_numpy()
            model = cat_model(seed * 100 + fold)
            model.fit(X.iloc[fit], target_fit, cat_features=cat_features, sample_weight=fit_weight,
                      eval_set=(X.iloc[valid], target_valid), verbose=False)
            fold_oof[valid] = restore(model.predict(X.iloc[valid]), train.iloc[valid])
            fold_test += restore(model.predict(X_test), test) / N_SPLITS
        oof += fold_oof / len(RANDOM_SEEDS)
        test_pred += fold_test / len(RANDOM_SEEDS)
    print(f'{kind}: CV RMSE = {root_mean_squared_error(y, oof):.4f}')
    return oof, test_pred

oof_direct, pred_direct = train_family('direct')
oof_demand, pred_demand = train_family('demand')
oof_store, pred_store = train_family('store_norm')

CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=0]="PRD-PRFP9S": Cannot convert 'PRD-PRFP9S' to float

In [4]:
# Conservative fallback ensemble. Its weights are chosen before looking at test labels.
fallback_pred = 0.40 * pred_demand + 0.25 * pred_direct + 0.35 * pred_store
fallback_oof = 0.40 * oof_demand + 0.25 * oof_direct + 0.35 * oof_store
print('Fallback CV RMSE:', root_mean_squared_error(y, fallback_oof))

NameError: name 'pred_demand' is not defined

## Exact v47 reproduction

The best public submission was a stable blend of four of the owner’s previously scored prediction files. The weights below reproduce its calculation exactly when those files are attached. Otherwise the fresh cross-validated fallback above is used.

In [5]:
component_files = {'v22': 'v22.csv', 'v36': 'v36.csv', 'v37': 'v37.csv', 'v42': 'v42.csv'}
exact_available = all((COMPONENT_DIR / filename).exists() for filename in component_files.values())

if exact_available:
    c = {name: pd.read_csv(COMPONENT_DIR / filename) for name, filename in component_files.items()}
    reference = c['v22']['id']
    assert all(list(frame.columns) == ['id', 'total_sales'] and len(frame) == len(test) for frame in c.values())
    assert all(frame['id'].equals(reference) for frame in c.values()), 'Component IDs must be aligned'
    v46 = (0.2437599522860421 * c['v22']['total_sales'].to_numpy() +
           0.31000588636381204 * c['v36']['total_sales'].to_numpy() +
           0.4462341613501459 * c['v42']['total_sales'].to_numpy())
    final_pred = 0.9966157383893812 * v46 + 0.0033842616106188207 * c['v37']['total_sales'].to_numpy()
    final_ids = reference
    method = 'Exact v47 scored blend'
else:
    final_pred = fallback_pred
    final_ids = test['id']
    method = 'Fresh cross-validated fallback ensemble'

print(method)

NameError: name 'fallback_pred' is not defined

In [ ]:
submission = pd.DataFrame({'id': final_ids, 'total_sales': np.maximum(final_pred, 0)})
assert len(submission) == len(test)
assert submission['id'].is_unique
assert submission['total_sales'].notna().all()
assert np.isfinite(submission['total_sales']).all()

submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print('Method:', method)
print('Saved:', submission_path)
display(submission.head())